In [17]:
import clickhouse_connect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

In [7]:
# Connect ke database
client = clickhouse_connect.get_client(
    host='localhost',
    port=8123,
    username='mahasiswa',
    password='bigdata123',
    database='praktikum',
    secure=False
)

print("Connected:", client.server_version)

Connected: 26.3.9.8


In [9]:
# fungsi untuk mengambil dataset
def load_csv(path, table_name="ecommerce_events"):
    chunk_size = 1_000_000
    total_rows = 0

    for chunk in pd.read_csv(path, chunksize=chunk_size):
        chunk.columns = [
            'event_time', 'event_type', 'product_id', 'category_id',
            'category_code', 'brand', 'price', 'user_id', 'user_session'
        ]

        # type conversion
        chunk['event_time'] = pd.to_datetime(chunk['event_time'], utc=True)
        chunk['product_id'] = chunk['product_id'].astype('uint64')
        chunk['category_id'] = chunk['category_id'].astype('uint64')
        chunk['price'] = chunk['price'].astype('float64')
        chunk['user_id'] = chunk['user_id'].astype('uint64')

        # cleaning
        chunk['category_code'] = chunk['category_code'].fillna('')
        chunk['brand'] = chunk['brand'].fillna('')
        chunk['user_session'] = chunk['user_session'].where(
            chunk['user_session'].notna(), None
        )

        valid_types = ['view', 'cart', 'remove_from_cart', 'purchase']
        chunk = chunk[chunk['event_type'].isin(valid_types)]

        client.insert_df(table_name, chunk)

        total_rows += len(chunk)
        print(f"{path} -> inserted {total_rows}")

    print(f"Finished: {path}")

In [10]:
# build user profile
def build_user_profile():
    query = """
    SELECT
        user_id,
        countIf(event_type = 'view') AS view_count,
        countIf(event_type = 'cart') AS cart_count,
        countIf(event_type = 'purchase') AS purchase_count,
        sumIf(price, event_type = 'purchase') AS total_spend,
        uniqExact(category_code) AS distinct_categories,
        uniqExact(toYYYYMMDD(event_time)) AS active_days
    FROM ecommerce_events_clean
    GROUP BY user_id
    """

    df = client.query_df(query)
    df['total_spend'] = df['total_spend'].fillna(0)

    print(f"User profiles: {len(df):,}")
    return df

In [11]:
#Scaling
def scale_features(df, features):
    scaler = RobustScaler()
    X = df[features]
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler

In [12]:
# kmeans
def run_kmeans(X_scaled, df, n_clusters=4):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    df['cluster'] = kmeans.fit_predict(X_scaled)
    return df, kmeans

In [13]:
#GMM
def run_gmm(X_scaled, df, n_components=4):
    gmm = GaussianMixture(n_components=n_components, random_state=42)
    df['cluster_gmm'] = gmm.fit_predict(X_scaled)
    return df, gmm

In [14]:
#evaluasi

In [15]:
def evaluate_model(X_scaled, labels, name="Model"):
    db = davies_bouldin_score(X_scaled, labels)
    sil = silhouette_score(X_scaled, labels)

    print(f"{name} -> DB: {db:.4f} | Silhouette: {sil:.4f}")
    return db, sil

In [18]:
# PCA Visualization
def run_pca(X_scaled, labels_kmeans, labels_gmm):
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)

    df_vis = pd.DataFrame({
        'pca_1': X_pca[:, 0],
        'pca_2': X_pca[:, 1],
        'cluster_kmeans': labels_kmeans,
        'cluster_gmm': labels_gmm
    })

    # =========================
    # Plot KMeans
    # =========================
    plt.figure()
    plt.scatter(df_vis['pca_1'], df_vis['pca_2'], c=df_vis['cluster_kmeans'])
    plt.title("PCA - KMeans Clustering")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.show()

    # =========================
    # Plot GMM
    # =========================
    plt.figure()
    plt.scatter(df_vis['pca_1'], df_vis['pca_2'], c=df_vis['cluster_gmm'])
    plt.title("PCA - GMM Clustering")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.show()

    print("Visualization done.")

In [ ]:
# Main Pipeline
if __name__ == "__main__":

    load_csv("E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv")
    load_csv("E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv")

    df_user = build_user_profile()

    features = [
        'view_count', 'cart_count', 'purchase_count',
        'total_spend', 'distinct_categories', 'active_days'
    ]

    X_scaled, scaler = scale_features(df_user, features)

    # KMeans
    df_user, kmeans = run_kmeans(X_scaled, df_user)

    # GMM
    df_user, gmm = run_gmm(X_scaled, df_user)

    # Evaluation
    evaluate_model(X_scaled, df_user['cluster'], "KMeans")
    evaluate_model(X_scaled, df_user['cluster_gmm'], "GMM")

    # PCA
    run_pca(
        X_scaled,
        df_user['cluster'],
        df_user['cluster_gmm']
    )

    print("\nDone. Everything is clean now.")


E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv -> inserted 1000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv -> inserted 2000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv -> inserted 3000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv -> inserted 4000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv -> inserted 5000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv -> inserted 6000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv -> inserted 7000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv -> inserted 8000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv -> inserted 9000000
